# T2SMark full-main 真实复现入口

该 Notebook 用于 Colab 冷启动: 挂载 Google Drive、拉取仓库、加载 SD3.5 Medium 所需依赖、运行 T2SMark full-main 官方路径、生成统一 adapter observations、构造正式导入候选记录并打包保存到 Google Drive。

正式逻辑位于 `paper_workflow/colab_utils/t2smark_full_main_reproduction.py`, Notebook 只作为远程执行入口。


## 运行前准备

1. 在 Colab 中选择 GPU runtime。
2. 确认 Hugging Face 账号已获得 `stabilityai/stable-diffusion-3.5-medium` 访问权限, 并配置 `HF_TOKEN`。
3. 默认读取 `configs/paper_main_full_prompts.txt`; `SLM_WM_T2SMARK_FULL_MAIN_PROMPT_LIMIT=0` 表示使用全部 full-main prompt。
4. 本入口会写出正式导入候选记录, 但只有 fixed-FPR 与攻击矩阵边界同时满足时, validator 才会接受为正式主表结果。


In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')
os.environ.setdefault('SLM_WM_T2SMARK_FULL_MAIN_DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/SLM/t2smark_full_main_reproduction')
os.environ.setdefault('SLM_WM_T2SMARK_MODEL_ID', 'stabilityai/stable-diffusion-3.5-medium')
os.environ.setdefault('SLM_WM_T2SMARK_FULL_MAIN_RUN_NAME', 't2smark_sd35_medium_full_main')
os.environ.setdefault('SLM_WM_T2SMARK_FULL_MAIN_PROMPT_LIMIT', '5')
os.environ.setdefault('SLM_WM_T2SMARK_FULL_MAIN_REQUIRE_CUDA', '1')


In [ ]:
%pip install -q --upgrade diffusers transformers accelerate safetensors sentencepiece protobuf huggingface_hub open_clip_torch scikit-learn scipy pandas datasets tqdm


In [ ]:
import importlib.metadata as importlib_metadata


def package_version_or_missing(package_name):
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return 'not_installed'


dependency_report = {
    'diffusers': package_version_or_missing('diffusers'),
    'transformers': package_version_or_missing('transformers'),
    'accelerate': package_version_or_missing('accelerate'),
    'huggingface_hub': package_version_or_missing('huggingface_hub'),
    'torch': package_version_or_missing('torch'),
}
dependency_report


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print('workspace_ready', workspace_dir)


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('huggingface_login_ready')
else:
    print('HF_TOKEN_not_found')


In [ ]:
import torch

device_report = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
print(device_report)
assert device_report['cuda_available'], '需要 Colab GPU runtime'


In [ ]:
from paper_workflow.colab_utils.t2smark_full_main_reproduction import run_default_t2smark_full_main_reproduction_plan

t2smark_full_main_summary = run_default_t2smark_full_main_reproduction_plan(root='.')
t2smark_full_main_summary


In [ ]:
from pathlib import Path

summary_path = Path('outputs/t2smark_full_main_reproduction/t2smark_full_main_reproduction_summary.json')
validation_path = Path('outputs/t2smark_full_main_reproduction/t2smark_full_main_formal_import_validation_report.json')
print(summary_path.read_text(encoding='utf-8') if summary_path.exists() else t2smark_full_main_summary)
if validation_path.exists():
    print(validation_path.read_text(encoding='utf-8'))


In [ ]:
import os
import subprocess
from datetime import datetime, timezone

from paper_workflow.colab_utils.t2smark_full_main_reproduction import package_t2smark_full_main_reproduction_outputs


def resolve_short_commit():
    try:
        result = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True, capture_output=True, text=True)
    except Exception:
        return 'git_unknown'
    return result.stdout.strip() or 'git_unknown'


utc = datetime.now(timezone.utc).strftime('%Y%m%dt%H%M%S%fZ').lower()
short_commit = resolve_short_commit()
archive_name = f't2smark_full_main_reproduction_package_{utc}_{short_commit}.zip'
drive_output_dir = os.environ.get('SLM_WM_T2SMARK_FULL_MAIN_DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/SLM/t2smark_full_main_reproduction')
archive_record = package_t2smark_full_main_reproduction_outputs(
    root='.',
    drive_output_dir=drive_output_dir,
    archive_name=archive_name,
)
archive_record.to_dict()


In [ ]:
from pathlib import Path

for result_path in sorted(Path('outputs/t2smark_full_main_reproduction').glob('*.json')):
    if result_path.is_file():
        print(result_path)
        print(result_path.read_text(encoding='utf-8')[:4000])

assert t2smark_full_main_summary['run_decision'] == 'pass', t2smark_full_main_summary
print('drive_archive_path', archive_record.drive_archive_path)
